# Домашнее задание по теме «Ансамбли алгоритмов»

Сравним Isolation Forest, Random Forest и родственные ансамбли на Wine Quality. Проверим фильтрацию train, исследуем влияние `max_features`, реализуем свой лес регрессии и оценки важности признаков.

В заданиях качество и параметры сравниваем по validation. Test сохраняем для однократной итоговой оценки выбранного решения.

**Максимум за работу: 10 баллов.**

Задачи 1–5: 2 + 1,5 + 3 + 2 + 1,5 балла. Баллы за подпункты входят в стоимость задачи.


## Разведочный анализ данных (EDA)

Ранее на семинаре мы рассматривали поведение ансамблей на задаче бинарной классификации в датасете Rain in Australia. Теперь реализуй структуру ансамблей в задаче регрессии.

В этом домашнем задании мы снова используем датасет Wine Quality. В нём два отдельных файла: около 1600 записей для красного вина и почти 4900 для белого. На каждый образец приходится 11 физических и химических характеристик (фиксированная кислотность, летучая кислотность, цитрат, остаточный сахар, хлориды, свободный и общий диоксид серы, плотность, pH, сульфаты и содержание алкоголя) и целевая метка `quality` — оценка дегустатора от 0 до 10 (на практике в диапазоне ~ 3–9).

Проверим на этом датасете, улучшает ли фильтрация train с помощью Isolation Forest качество регрессии.

Импортируй библиотеки и [загрузи Wine Quality](https://archive.ics.uci.edu/dataset/186/wine+quality). Используй его для задачи регрессии и предскажи переменную `quality`.

In [ ]:
!pip install tqdm

In [ ]:
# Импорт библиотек
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    VotingRegressor, BaggingRegressor,
    RandomForestRegressor, ExtraTreesRegressor, IsolationForest)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from time import time

In [ ]:
# Предупреждения оставляем видимыми.


Теперь загрузи датасет и проведи разведочный анализ данных (EDA) и препроцессинг.

In [ ]:
# 1. Ссылки на CSV
red_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
white_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"

# 2. Считываем в DataFrame, указывая разделитель ';'
df_red = pd.read_csv(red_url, sep=';')
df_white = pd.read_csv(white_url, sep=';')

# 3. Добавляем колонку с типом вина
df_red["type"] = "red"
df_white["type"] = "white"

# 4. Объединяем
df = pd.concat([df_red, df_white], ignore_index=True)

# Точные повторы не должны попасть одновременно в train и отложенные части.
print('Строк в исходных CSV:', len(df))
print('Точных повторов:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print('Строк после удаления точных повторов:', len(df))
df.head()


In [ ]:
df['type'].value_counts()

In [ ]:
# Нет пропусков, один категориальный признак
df.info()

In [ ]:
# Проверяем диапазоны и редкие значения
df.describe().T

In [ ]:
# Работаем с категориальной фичей
df = pd.get_dummies(df, columns=["type"], drop_first=True, dtype='int')  # Появится столбец type_white
df.head()

In [ ]:
# Точки за усами boxplot требуют интерпретации
num_cols = df.select_dtypes(include="number").columns
for col in num_cols:
    plt.figure()
    plt.boxplot(df[col])
    plt.title(col)
    plt.ylabel(col)
    plt.show()

In [ ]:
X = df.drop('quality',axis=1)
y = df['quality']

In [ ]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval)
X_train, X_valid, X_test = X_train.copy(), X_valid.copy(), X_test.copy()
print(f'train: {len(X_train)}, validation: {len(X_valid)}, test: {len(X_test)}')


In [ ]:
numeric = X.drop(columns='type_white').select_dtypes(include=['number']).columns
scaler = StandardScaler()
X_train[numeric] = scaler.fit_transform(X_train[numeric])
X_valid[numeric] = scaler.transform(X_valid[numeric])
X_test[numeric] = scaler.transform(X_test[numeric])


Некоторые значения сильно отличаются от основной массы. Например, высокий `residual sugar` требует проверки единиц и источника данных. Редкость или удалённость на boxplot сама по себе не доказывает ошибку наблюдения.


Isolation Forest выделяет необычные сочетания признаков. Проверим, улучшает ли удаление таких объектов из train качество линейной регрессии на неизменной validation.


### Задача 1. Изолированный лес [2 балла]

Проверь гипотезу о полезности фильтрации train с помощью Isolation Forest. Аномалия по признакам не обязательно является ошибкой или неинформативным объектом.

- Создай `contamination_array` из значений от 0.001 до 0.1 с шагом 0.001.
- Обучи `lr_orig` на `X_train`, `y_train`. Посчитай MSE на `X_valid`, `y_valid` и сохрани в `mse_orig`. Это вариант без удаления. [0,5 балла]
- Обучи один `IsolationForest(n_estimators=300, contamination='auto', random_state=42, n_jobs=-1)` на `X_train`. Сохрани `scores = iso.score_samples(X_train)`.
- Для каждого `c` из `contamination_array` найди порог `np.quantile(scores, c)`. Оставь train с `scores >= threshold`, обучи на нём `lr_clean` и добавь MSE на той же validation в `mse_clean`. [1 балл]
- Построй график `mse_clean` и горизонтальную линию `mse_orig`. [0,5 балла]
- Выбери лучшую долю удаления, учитывая вариант 0. Объясни результат. Позволяет ли этот опыт установить наличие ошибочных наблюдений?

При фиксированных данных и seed числовой `contamination` меняет порог, а не деревья. Поэтому достаточно одного обучения Isolation Forest. Из-за совпадающих скоров доля удалённых строк может немного отличаться от заданной. Test в подборе не используется.


In [ ]:
# Напиши код здесь
# contamination_array = ...
# lr_orig = ...
# y_pred_orig = ...
# mse_orig = ...
# iso = ...
# scores = ...
# mse_clean = []
# for c in tqdm(contamination_array):
#     threshold = ...
#     keep = ...
#     X_train_clean, y_train_clean = ...
#     lr_clean = ...
#     y_pred_clean = ...
#     mse_clean.append(...)
# best_contamination = ...  # учти вариант без удаления


In [ ]:
# Построй график
plt.figure(figsize=(12, 8))

plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
# Напиши код здесь
# plt.plot(...)
# plt.plot(...)
plt.xlabel('contamination', fontsize=16)
plt.ylabel('Validation MSE', fontsize=16)
plt.legend()
plt.tick_params(axis='y')

plt.grid(True)
plt.show()

In [ ]:
# 1. Считаем матрицу корреляций
corr = df.loc[X_train.index].corr(numeric_only=True)

# 2. Берём только столбец quality, убираем строку quality и сортируем
corr_q = corr["quality"].drop("quality").sort_values(ascending=False)

# 3. Рисуем теплокарту
plt.figure(figsize=(6, len(corr_q) * 0.4))
sns.heatmap(corr_q.to_frame(), annot=True, cmap="vlag", cbar=False)
plt.title("Корреляция признаков с quality на train")
plt.ylabel("Признак")
plt.xlabel("Correlation")
plt.show()

## Random Forest и родственные ансамбли

### Задача 2. Корреляция базисов и метрики [1,5 балла]

Исследуй влияние `max_features` на MSE случайного леса и сходство прогнозов его деревьев. Маленькое число кандидатов может сделать деревья менее похожими и одновременно ослабить каждый базис. Итоговый эффект проверим на validation.

Для каждого целого `max_features` от 1 до числа признаков:

1. Обучи `RandomForestRegressor` со 100 деревьями, текущим `max_features`, `random_state=42` и `n_jobs=-1` на исходных `X_train`, `y_train` без фильтрации.
2. Добавь MSE на `X_valid`, `y_valid` в `valid_mses`. [0,5 балла]
3. Собери прогнозы деревьев на validation в `preds` формы `(100, n_valid)`. Используй `rf.estimators_`. Для отдельного дерева передавай `X_valid.to_numpy()`, поскольку деревья леса обучаются без имён столбцов.
4. Посчитай `corr_matrix = np.corrcoef(preds)`.
5. Выбери верхний треугольник без диагонали с помощью `np.triu_indices_from(corr_matrix, k=1)`.
6. Добавь среднюю корреляцию в `mean_corrs`. [1 балл]

Это корреляция прогнозов по разным объектам при фиксированном обучении. Она не совпадает с корреляцией по случайности обучения в формуле дисперсии бэггинга. Сопоставление кривых показывает наблюдаемую связь, а не доказывает причинное влияние корреляции на MSE.

Справка: [corrcoef](https://numpy.org/doc/stable/reference/generated/numpy.corrcoef.html), [triu_indices_from](https://numpy.org/doc/stable/reference/generated/numpy.triu_indices_from.html).


In [ ]:
# Варианты max_features
feature_settings = range(1, X_train.shape[1]+1)

mean_corrs = []
valid_mses = []

for mf in feature_settings:
    # Напиши код здесь
    # rf = ...
    # ...
    # y_pred = ...
    # valid_mses....

    # Собери предсказания каждого дерева на валидации (n_trees × n_samples)
    # preds = np.array([...])

    # Матрица корреляций (n_trees × n_trees)
    # corr_matrix = ...

    # Возьми только верхний треугольник без диагонали
    # triu = corr_matrix[...]
    # mean_corrs....

Построй график и посмотри, что получилось.

In [ ]:
# --- График ---
fig, ax1 = plt.subplots(figsize=(8,6))

ax1.plot(feature_settings, mean_corrs, marker='o', color='tab:blue', label="Mean correlation")
ax1.set_xlabel("max_features")
ax1.set_ylabel("Mean correlation", color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.plot(feature_settings, valid_mses, marker='s', color='tab:red', label="Validation MSE")
ax2.set_ylabel("Validation MSE", color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')

plt.title("Correlation between trees vs Validation MSE")
fig.tight_layout()
plt.show()

**Вопросы.**

- При каком `max_features` минимальна валидационная MSE?
- Как меняется средняя корреляция прогнозов деревьев при росте `max_features`?
- Монотонна ли каждая кривая? Совпадает ли минимум MSE с минимумом корреляции?

Объясни наблюдения через качество отдельных деревьев и сходство их прогнозов. Малое число признаков может одновременно уменьшить корреляцию и ухудшить качество каждого дерева.

В следующей задаче сравним разные способы случайного отбора строк и признаков. Изменение этих правил влияет и на отдельные деревья, и на сходство их прогнозов. Сравним итоговую MSE на общей validation.

### Задача 3. Сравнение ансамблей [3 балла]

Сравни шесть ансамблей деревьев и Voting по валидационной MSE и времени. Все модели обучай на исходном train без фильтрации из задачи 1.

Инициализируй модели с `random_state=rs`, `n_jobs=-1` и `warm_start=True`. В качестве базиса Bagging используй неограниченное `DecisionTreeRegressor`. [1,5 балла]

| Модель | Доля строк | Возвращение строк | Признаки |
|---|---|---|---|
| Bagging | `sample_frac` | Да | Все |
| Pasting | `sample_frac` | Нет | Все |
| Random Subspaces | 1.0 | Нет | `feature_frac` на всё дерево |
| Random Patches | `sample_frac` | Нет | `feature_frac` на всё дерево |
| Random Forest | `sample_frac` | Да | `feature_frac` на каждый узел |
| Extra Trees | `sample_frac` | Да | `feature_frac` на каждый узел |

Первые четыре реализуй через `BaggingRegressor`. Признаки выбирай без возвращения (`bootstrap_features=False`). У Random Forest и Extra Trees `max_samples` используется вместе с `bootstrap=True`.

Создай `VotingRegressor` из линейной регрессии, kNN с 15 соседями и SVR с `C=3.0`, `epsilon=0.1`. Обучи его на train, оцени на validation. Сохрани MSE в `voting_mse`, время обучения и прогнозирования в `voting_time`. [0,5 балла]

Для каждого `n` из `n_values` задай моделям `n_estimators=n`, вызови `.fit()`, получи прогноз на validation и запиши MSE и время в `mse_logs`, `time_logs`. [1 балл]

С `warm_start` на каждом шаге обучаются только новые деревья. Измеренное время включает их обучение и прогноз всем текущим ансамблем. Данные и остальные настройки в цикле должны оставаться неизменными.


In [ ]:
# ==== Гиперпараметры ====
start_n = 12        # начальное количество базисов
step_n = 12         # шаг на итерацию
rounds = 15         # число проверяемых размеров ансамбля, до 180
sample_frac = 0.7   # доля объектов на базовую модель
feature_frac = 0.6  # доля признаков, смысл зависит от ансамбля
warm_start=True
rs = 42             # random_state

# n значений
n_values = list(range(start_n, start_n + rounds*step_n, step_n))

# Напиши код здесь

# ==== Базовая модель ====

# base_tree = ...

# ==== Ансамбли ====

# bagging = ...

# pasting = ...

# subspaces = ...

# patches = ...

# rf = ...

# extratrees = ...

models = {
    "bagging":    bagging,
    "pasting":    pasting,
    "subspaces":  subspaces,
    "patches":    patches,
    "rf":         rf,
    "extratrees": extratrees,
}

# Напиши код здесь

# ==== VotingRegressor (независим от n_estimators) ====
# voting = ...

# t0 = ...
# ...
# y_pred_v = ...
# t1 = ...
# voting_mse = ...
# voting_time = ...

# === Логи ===
mse_logs = {name: [] for name in models}
time_logs = {name: [] for name in models}
# добавим ключи для voting (заполним после цикла константами)
mse_logs["voting"] = []
time_logs["voting"] = []

# Напиши код здесь

# === Цикл по num_estimators ===
for n in tqdm(n_values):
    # обучаем и мерим для каждого ансамбля
    for name, mdl in models.items():
        # mdl.set_params(n_estimators=n)
        # t0 = ...
        # ...
        # y_pred = ...
        # t1 = ...

        # mse_logs[name]....
        # time_logs[name]....

        pass

Сопоставь графики полученных валидационных MSE и времени обучения + предсказания.

In [ ]:
mse_logs["voting"] = [voting_mse] * len(n_values)
time_logs["voting"] = [voting_time] * len(n_values)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name in mse_logs:
    axes[0].plot(n_values, mse_logs[name], marker='o', label=name)
    axes[1].plot(n_values, time_logs[name], marker='o', label=name)
axes[0].set_ylabel('Validation MSE')
axes[1].set_ylabel('Время обучения новых деревьев и прогноза, с')
for ax in axes:
    ax.set_xlabel('n_estimators')
    ax.grid(True)
axes[1].legend(loc='upper left', bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()


**Вопросы.** Какой ансамбль получил наименьшую MSE на validation? Сравни время одной итерации и суммарное время последовательного обучения до выбранного числа деревьев.

У Voting горизонтальная линия времени показывает одно полное обучение и прогноз. У древесных ансамблей с `warm_start` каждая точка включает только обучение новых деревьев и прогноз всем ансамблем. Эти времена нельзя напрямую трактовать как стоимость обучения моделей с нуля.

### Задача 4. Свой класс леса регрессии [2 балла]

Реализуй методы класса ниже. На корректный отбор строк и построение деревьев приходится 1 балл, на усреднение прогнозов и `score` 0,5 балла. `max_features` передавай в каждое `DecisionTreeRegressor`: набор кандидатов выбирается в каждом узле, а не один раз на дерево.

- `_draw_indices` возвращает индексы строк. При целочисленном `max_samples` размер равен ему. При доле используй `max(1, round(max_samples * n_rows))`. С возвращением выбирай при `bootstrap=True`, без возвращения при `False`.
- При каждом `fit` сбрасывай список деревьев и генератор `np.random.RandomState(self.random_state)`. Для каждого дерева выбирай строки и отдельный целочисленный seed, затем обучай его на всех столбцах выбранных строк.
- `predict` усредняет прогнозы деревьев.
- `score` возвращает R², как у регрессоров sklearn. MSE считай отдельно через `mean_squared_error`.

Проверь допустимость числа деревьев, размера подвыборки и вызова `predict` до обучения. В этой учебной реализации разрешён частичный отбор строк без возвращения. У sklearn Random Forest параметр `max_samples` доступен только при `bootstrap=True`.

`BaseEstimator` и `RegressorMixin` обеспечивают совместимость со sklearn при расчёте permutation importance. Для `score` используй импортированную функцию `r2_score`.


In [ ]:
class MyRandomForestRegressor(RegressorMixin, BaseEstimator):
    def __init__(self, n_estimators=100, max_samples=1.0, bootstrap=True,
                 max_features='sqrt', random_state=None, base_estimator_params=None):
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.bootstrap = bootstrap
        self.max_features = max_features
        self.random_state = random_state
        self.base_estimator_params = base_estimator_params
        self.estimators_ = []
        self.rng_ = np.random.RandomState(random_state)

    def _draw_indices(self, n_rows):
        m = self.max_samples
        # size: для int берём m, для доли max(1, round(m * n_rows))
        # bootstrap=True  -> self.rng_.randint(0, n_rows, size)
        # bootstrap=False -> self.rng_.choice(n_rows, size=size, replace=False)
        # your code

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        # Сбрось RNG для воспроизводимого повторного fit
        # self.rng_ = ...
        self.n_samples_ = # your code
        self.estimators_ = # your code

        for _ in range(self.n_estimators):
            # your code: fit each base model in ensemble

        return self

    def predict(self, X):
        X = np.asarray(X)
        # your code

    def score(self, X, y):
        # your code: calculate R² for (X, y)

Сравни свою реализацию с sklearn при одинаковых параметрах из задачи 3: 180 деревьев, `max_samples=0.7`, `max_features=0.6`, неограниченная глубина, бутстрэп, seed 42. Оба варианта оцени на одной validation.

Равный seed не требует точного совпадения прогнозов: порядок генерации случайных чисел в реализациях может различаться. Сравни принцип работы и величины MSE.


In [ ]:
rf = MyRandomForestRegressor(
    n_estimators=180,
    max_samples=0.7,
    bootstrap=True,
    max_features=0.6,
    random_state=42,
    base_estimator_params={"max_depth": None, "min_samples_leaf": 1}
)

rf.fit(X_train, y_train)

In [ ]:
# MSE нашего класса
y_pred = rf.predict(X_valid)
print("My RF MSE :", mean_squared_error(y_valid, y_pred))

In [ ]:
# MSE класса sklearn
print("Sklearn RF MSE :", mse_logs['rf'][-1])

**Вопрос.** Сравни MSE реализаций при seed 42. Затем повтори свою модель с seed 45. Как меняется разница? Почему один запуск не позволяет установить превосходство реализации? Сравнение при двух seed и объяснение: 0,5 балла.


In [ ]:
# Создай отдельную модель rf_seed45 с теми же параметрами и random_state=45.
# Обучи на X_train, y_train и сравни MSE на той же validation.
# Сохрани модель rf с seed 42 для сравнения.


### Задача 5. Важности признаков и итоговая оценка [1,5 балла]

Создай подкласс `MyRandomForestWithImportance` и добавь свойство `feature_importances_` с помощью `@property`. Исходный класс из задачи 4 оставь без изменений.

- Усредни нормированные `tree.feature_importances_` по `self.estimators_` и нормируй результат на сумму. Если все деревья без разбиений, верни нулевой вектор.
- До обучения, когда список деревьев пуст, верни `None`.

Это MDI. Вектор имеет по одному элементу на каждый исходный признак, в порядке столбцов train.

Корректная реализация MDI и её график: 0,5 балла.


In [ ]:
class MyRandomForestWithImportance(MyRandomForestRegressor):
    @property
    def feature_importances_(self):
        # Напиши код здесь
        pass

Реализуй MDI на нашем датасете и посмотри, что получилось.

In [ ]:
rf = MyRandomForestWithImportance(
    n_estimators=180,
    max_samples=0.7,
    bootstrap=True,
    max_features=0.6,
    random_state=42,
    base_estimator_params={"max_depth": None, "min_samples_leaf": 1}
)

rf.fit(X_train, y_train)

In [ ]:
fi = # таблица column и feature_importance
fi = fi.sort_values('feature_importance')
plt.barh(fi['column'], fi['feature_importance'])
plt.xlabel('MDI, нормированная важность')
plt.tight_layout()
plt.show()


Вызови sklearn `permutation_importance` для своей обученной модели на `X_valid`, `y_valid` с `scoring='neg_mean_squared_error'`, 10 повторами и `random_state=42`.

При таком скорере положительная важность равна увеличению MSE после перемешивания. Она имеет единицы MSE и не должна совпадать по масштабу с нормированным MDI. Функция не обучает модель заново.

На графике покажи средние важности и стандартные отклонения `result.importances_std` по повторам.

Вызов permutation importance, график и интерпретация: 0,5 балла.


In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    # your code
    # estimator, X_valid, y_valid,
    n_repeats=10, random_state=42,
    scoring='neg_mean_squared_error'
)

fi = # таблица column, feature_importance и std из result
fi = fi.sort_values('feature_importance')
plt.barh(fi['column'], fi['feature_importance'], xerr=fi['std'])
plt.xlabel('Увеличение MSE при перемешивании')
plt.tight_layout()
plt.show()

**Вопрос.** Сравни ранжирование признаков по MDI и permutation importance. Почему масштабы различаются? Как коррелированные признаки и разброс по перестановкам могут влиять на вывод?

После всех сравнений зафиксируй выбранную модель и настройки по validation. Для итоговой оценки переобучи скейлер и модель на объединённых train и validation, затем один раз посчитай MSE на test. Если выбрана фильтрация, обучи Isolation Forest только на объединённом train и validation и фильтруй только эту обучающую часть. Test сохраняй полностью. Выбор модели, итоговое переобучение и оценка на test: 0,5 балла.


In [ ]:
# Выбери модель и настройки по validation.
# Возьми сырые X.loc[X_trainval.index] и X.loc[X_test.index].
# Обучи новый скейлер на объединённых train и validation.
# Преобразуй обе части, переобучи выбранную модель и один раз оцени test MSE.
